# 05 — Module Guide

RLDX-1 is built from several opt-in modules that plug into the
base Qwen3-VL backbone + MSAT action model. You can enable each
independently from the training CLI. This notebook covers:

1. **Video** — multi-frame vision history via video token compression.
2. **Memory** — temporal context aggregation on cognition tokens.
3. **Motion module** — motion-aware spatio-temporal summarization inside the vision encoder.
4. **Physics** — tactile / torque conditioning stream for the action head.
5. **Cognition tokens** — the learned tokens that carry VL → action information.
6. **Combining modules** — how the opt-in pieces stack.

Each section is self-contained: read the sections you care about
and ignore the rest. The examples use the canonical robocasa
finetune command from [`03_finetuning.ipynb`](03_finetuning.ipynb) as
a baseline and only show the *incremental* flags for each module.

All module flags live on `TrainConfig` in `rldx/configs/train_config.py`.


## 1. Video module

**What it is.** A history of RGB frames is fed to the vision
encoder instead of a single current frame. Internally this is
routed through the `vtc_qwen3_vl` backbone, which extends
Qwen3-VL to consume `video_length` frames per observation.

**When to use.** Any task where the current frame is ambiguous
without recent history (flipping, pouring, sequential sub-goals).

**Flags.** Video is **always on**; only the shape parameters are CLI-tunable.


| Flag | Default | Notes |
|---|---|---|
| `--video-length` | `4` | Number of past frames (including current). |
| `--video-stride` | `2` | Stride between frames. With L=4, S=2 the launcher inserts deltas `{-6, -4, -2, 0}` into the modality config's `video` entry. |

**Dataset requirement.** Your LeRobot dataset must ship the video
mp4 files — the `meta/modality.json` must declare a `video` entry.
At training time, the launcher extends the modality config's
`video.delta_indices` with the striding schedule above so the dataset
loader pulls history frames alongside the current frame.

**Inference.** The client must always supply `T = video_length` frames
per video key in the observation, in chronological order. With
`--use-sim-policy-wrapper` this shape check is enforced (see the
validation code in `rldx_policy.py:368`).


### Example — add video to the robocasa fine-tune


In [ ]:
video_flags = [
    '--video-length', '4',
    '--video-stride', '2',
]
print('add to launch_train.py:', video_flags)
# Combined with --base-model-path RLWRLD/RLDX-1-PT and
# --modality-config-path rldx/configs/data/robocasa_config.py, this
# gives the 4-frame history used in the canonical training run.


## 2. Memory module

**What it is.** A small Transformer that consumes the cognition tokens from the past `memory_length` timesteps and produces
temporally-aggregated cognition-token embeddings for the action head.
Think of it as 'KV-cache with learned fusion' over cognition tokens,
replacing or augmenting raw per-step cognition tokens. Implementation
lives in `rldx/model/modules/memory.py`.

**When to use.** Long-horizon tasks that need more temporal
context than a single video window can carry (e.g. 'I was
stirring 3 seconds ago, now I must plate').

**Flags.**

| Flag | Default | Notes |
|---|---|---|
| `--use-memory` | `False` | Master switch. |
| `--memory-length` | `4` | Past timesteps. The launcher adds memory anchors `{-(memory_length-1-i) * memory_stride for i in range(memory_length)}` to `video.delta_indices`. |
| `--memory-stride` | `16` | Stride (in action-step units) between consecutive memory anchors. |
| `--memory-n-cog-tokens` | `None` (= `n_cog_tokens`) | Must be `<= n_cog_tokens`. Subset of cognition tokens that pass through the memory module. |
| `--concat-memory` | `False` | If `True`, concat memory-augmented tokens *after* the originals instead of replacing. Doubles the cognition-token length reaching the action head. |
| `--blockwise-attn-for-memory` | `False` | Swap token-level causal attn for block-wise causal attn (bidirectional within a timestep). |
| `--memory-dropout-prob` | `0.0` | Dropout on memory-augmented tokens. Requires `--concat-memory`. |

**Dataset requirement.** The dataset must be long enough to
populate `memory_length × memory_stride` past steps. Shorter episodes are
padded via `run_config.data.allow_padding = True` (set
automatically when `--use-memory` is active).

**Inference.** `RLDXPolicy` maintains a per-session memory cache.
Pass `session_ids` and `reset_memory` inside `options` when calling
`get_action(obs, options={...})` — reset at episode boundaries.
See the memory handling at `rldx/policy/rldx_policy.py:225+`.

**Running a memory-trained model without memory.** If you want to
A/B a memory checkpoint against its memoryless baseline without
re-training, start the server with `--deactivate-memory`. The
loader skips the memory module weights and runs the base RLDX.


### Example — memory on top of video


In [ ]:
memory_flags = [
    '--use-memory',
    '--memory-length', '4',
    # Optional: keep the full cognition-token count (omit --memory-n-cog-tokens).
    # Optional: concat instead of replace.
    # '--concat-memory',
    # '--memory-dropout-prob', '0.1',  # only with --concat-memory
]
print('add to launch_train.py (on top of video flags):', memory_flags)


## 3. Motion module

**What it is.** A Motion-Aware Spatio-Temporal Summarization
block injected inside the Qwen3-VL vision encoder. It computes
patch-level motion correlations across video frames and folds
them back into either (a) a residual add at an intermediate
vision-encoder layer, or (b) a projected token sequence prepended
to the LLM input. See `rldx/model/modules/backbone/motion.py`.

**When to use.** Manipulation tasks where fine-grained object
motion cues are important (pouring, pushing, tool use). Requires
the multi-frame video input that the release backbone always carries.

**Flags.**

| Flag | Default | Notes |
|---|---|---|
| `--use-motion` | `False` | Master switch. Motion correlations operate on the video tokens, which are always available. |
| `--motion-insert-layer` | `9` | Vision-encoder layer index at which motion module is injected. |
| `--motion-injection-point` | `vision_encoder` | `vision_encoder` = residual add at the insert layer. `vl_input` = project pooled motion-module tokens and prepend as LLM input. |
| `--motion-pool-type` | `avg` | For `vl_input` mode: `avg` (adaptive pool) or `conv` (learned depthwise separable conv). |
| `--motion-drop` | `True` | Drop motion-module tokens at internal-projection layer 4 to save compute. Setting `False` keeps them through all LLM layers (~40% more FLOPs). |
| `--motion-gradient-check` | `False` | Register backward hooks that log motion-module gradient norms — debugging only. |

**Dataset requirement.** Same as video — no new modality keys.
The motion module operates on the existing video frames.

**Inference.** Transparent. motion-module weights live inside the backbone
and load automatically via `from_pretrained`.


### Example — motion module on top of video


In [ ]:
motion_flags = [
    '--use-motion',
    '--motion-insert-layer', '9',
    '--motion-injection-point', 'vision_encoder',
    # '--motion-injection-point', 'vl_input',
    # '--motion-pool-type', 'conv',
]
print('add to launch_train.py:', motion_flags)


## 4. Physics module (tactile / torque)

**What it is.** A parallel conditioning stream that feeds
low-dimensional continuous sensors (tactile pressure, joint
torques, force-torque) into the MSAT action model alongside the
VL features. Encoder / decoder live in
`rldx/model/modules/action_model/{physics,physics_head}.py`.

**When to use.** Contact-rich manipulation where vision alone
can't disambiguate (grasp firmness, insertion, wiping pressure).

**Flags.**

| Flag | Default | Notes |
|---|---|---|
| `--use-physics` | `False` | Master switch. |
| `--physics-keys` | — | List of modality keys (e.g. `tactile torque`). Must be present as top-level keys in your modality config dict. |
| `--physics-dims` | — | Per-key dimensions. `len(physics-dims) == len(physics-keys)`. Total physics dim = sum. |
| `--physics-loss-weight` | `0.1` | Scalar multiplier on the physics prediction loss term. |
| `--allow-missing-physics` | `False` | Skip physics for embodiments that don't carry the listed keys. Useful in cross-embodiment mixes. |

**Dataset requirement.** Your modality config must declare each
physics key with its own `ModalityConfig(delta_indices=..., modality_keys=[...])`
entry at the top level (same layout as `state` / `action`). All
physics keys must share the same `delta_indices` — the launcher
asserts this at startup.


### Example — physics-aware fine-tune


In [ ]:
# Suppose your embodiment carries a 30-D tactile array and a 7-D
# torque vector. Your modality config would need:
#   my_cfg['tactile'] = ModalityConfig(delta_indices=[0], modality_keys=['tactile'])
#   my_cfg['torque']  = ModalityConfig(delta_indices=[0], modality_keys=['torque'])

physics_flags = [
    '--use-physics',
    '--physics-keys', 'tactile', 'torque',
    '--physics-dims', '30', '7',
    '--physics-loss-weight', '0.1',
    # If some datasets in a mix lack physics, allow them through:
    # '--allow-missing-physics',
]
print('add to launch_train.py:', physics_flags)


## 5. Cognition tokens

**What it is.** Learned query tokens appended to the Qwen3-VL
input sequence. After the backbone runs, these tokens carry the
VL-conditioned information that the MSAT action model attends to.
They are the *only* channel between the VLM and the action head,
so their count is a direct capacity knob.

**Flag.**

| Flag | Default | Notes |
|---|---|---|
| `--n-cog-tokens` | `64` | Model's public release default. Pre-trained checkpoints are trained at 64; deviating forces the `cog_emb` weight to be re-initialised (see `_load_state_dict_with_shape_filter`). |

**Dataset requirement.** None.

**Caveat.** Changing `n_cog_tokens` when fine-tuning from a
pre-trained checkpoint silently drops the pre-trained `cog_emb`
weight and replaces it with a fresh init. This usually costs
several points of downstream success rate. Only change it when
(a) pre-training from scratch, or (b) you accept the re-init.


## 6. Combining modules

All four opt-in modules (memory / motion / physics / cognition-token count) can be enabled independently and combined freely on top of the always-on video backbone. Motion module reads from the same video tokens the backbone consumes; memory replaces or augments the cognition tokens flowing into the action head; physics adds a parallel conditioning stream and is fully orthogonal to the rest.

- `--memory-stride` (default 16) is independent of `--video-stride`.
- The MSAT action model is the only action model shipped; no flag is needed to enable it.


## 7. Putting it all together

A kitchen-sink fine-tune command that enables everything looks
roughly like this:

```bash
uv run torchrun --nproc_per_node=8 --master_port=29500 \
    rldx/experiment/launch_train.py \
        --base-model-path RLWRLD/RLDX-1-PT \
        --dataset-path /path/to/dataset \
        --embodiment-tag GENERAL_EMBODIMENT \
        --modality-config-path rldx/configs/data/robocasa_config.py \
        --output-dir ckpt/rldx_kitchen_sink \
        --n-cog-tokens 64 \
        --video-length 4 --video-stride 2 \
        --use-memory --memory-length 4 \
        --use-motion --motion-insert-layer 9 --motion-injection-point vision_encoder \
        --use-physics --physics-keys tactile torque --physics-dims 30 7 \
        --color-jitter-params brightness 0.3 contrast 0.4 saturation 0.5 hue 0.08 \
        --global-batch-size 64 --learning-rate 1e-4 --max-steps 60000
```

The examples above use `RLWRLD/RLDX-1-PT` as the base checkpoint.
Not every task benefits from every module — start
with video, add memory / motion module / physics one at a time and measure.
